# Ocular adversarial XAI: anatomy-guided models and Anatomical Shapley Attribution

One Kaggle session (or several, see *Resume*) that trains the anatomy-guided heads of `src/models/anatomy_model.py` under the **cross-session protocol** with the recipe of `configs/anatomy.yml` (= `configs/cross_session.yml` plus the `anatomy` section, with an epoch cap of 40 instead of 30 because the new heads start slower; early stopping ends every run, so only the head differs from the cross-session baseline), evaluates every model (04 test accuracy, 15 verification P2 + P3), runs the **gradient-masking check** (24) on the mask models and the gap model, and explains the best mask-supervised model and the baseline part by part with **Anatomical Shapley Attribution** (22, analysed by 23).

22 runs with the settings `ASA_*` of cell 2: the attacks (by default fgsm, pgd and `aas_pgd`, the PGD through the attention-temperature-scaled model of `src/attacks/masking.py`), the verification pairs (a genuine and an impostor template per image), the KernelSHAP / LIME part baselines on the hiding methods of `ASA_BASELINE_HIDING` and the pixel baselines on the conditions of `ASA_PIXEL_CONDITIONS`. Every 22 run of the session (smoke C and both models) uses the same flags, so the smoke run times the full run, except that `aas_pgd` runs only on models with an attention pool (on the baseline it is pgd itself and would only duplicate every pgd condition). 24 tests whether gradient attacks fail because the model is robust or because its gradients are uninformative (PGD success against steps and budget, pgd vs `aas_pgd` vs random restarts, gradient and attention statistics); **no robustness difference between two models counts unless 24 reports no masking flag for both**.

Models (`ARCHS`, cell 2; each trained for every seed in `SEEDS`):
- `attn_mask`: one attention map pooled over the 8 x 8 feature cells, trained with CE + lambda x KL(eye-region target || map). Masks are used **only in training**.
- `parts`: one attention map per part group (iris, sclera, lid, brow, skin) and a linear embedding per part, summed: the margin-free cosine is exactly the sum of per-part terms (the *receipt*), with a per-part mask loss.
- `attn`, `gap`: the same heads without masks (controls for the mask loss and for attention itself).
- `attnet_gate`: AttNet's fixed gate (Zhao & Kumar, TIFS 2018: x5 inside the eye region, x1/5 outside). It needs the eye masks **at test time** too (04 / 15 `--region-masks`).
- `baseline` (optional): the vendored Xception + ArcFace. Usually the cross-session baseline of `train_cross_session.ipynb` is attached instead (`BASELINE_CKPT`).

**Before you start**
1. `BRANCH` (cell 1) must hold the anatomy build: 03 `--arch` / `--masks`, scripts 21, 22, 23 and 24, `configs/anatomy.yml`. Cell 6 stops if a script, flag or the config is missing.
2. Settings: accelerator **GPU T4 x2**, Internet **on** (to clone the repo and download the eye-mask weights `combine.pth` for 17 and 21), secret `github_token` attached.
3. Inputs: `mubashirulhassan00/visob-augmented-recent`; for 22's comparison model the output of `train_cross_session.ipynb` (set `BASELINE_CKPT`); to continue a stopped run, this notebook's earlier output (`PREVIOUS_OUTPUT`).
4. Keep this notebook **private**. `results.zip` holds no images and no masks; never upload it, the checkpoints or the masks to a public place.

**Workflow**
- First run all cells interactively with `RUN_FULL = False` (about 1 h): checks, the manifest and the cross-session split, eye masks of the 20-class smoke tree (21) and of the pool of verification images 22 draws from (17), a smoke training of every arch (2 epochs, 20 classes, both GPUs busy), a `--resume` test of the mask-loss trainer, smoke runs of 04 and 15 on every smoke model and of 24 on the smoke models of `MASKING_ARCHS`, a 22 + 23 smoke run, and the time estimate.
- Then set `RUN_FULL = True` and use **Save Version > Save & Run All**. The full run segments every image once (21, masks64 for train, validation, test and, for `attnet_gate`, the open set), trains the models queued over both GPUs (mask-free archs start while the masks are being made), evaluates each (and runs 24 on the models of `MASKING_ARCHS`), and runs 22 on the chosen mask model and the baseline when its estimate fits before `HARD_STOP_H`.
- Download `results.zip` from the Output panel. Never import TensorFlow in a notebook cell: every TensorFlow or PyTorch script runs in its own process.

| Cell | GPU 0 | GPU 1 | Time |
|---|---|---|---|
| 7. Data (CPU) | | | ~20 min (01, 02 cross-session, open set, 20-class tree, mask and ASA lists) |
| 9. Smoke A | 21 masks of the smoke tree, then 03 smoke of the mask archs | 17 masks of the ASA pool, then 03 smoke of the mask-free archs | ~30 min |
| 10. Smoke B | `--resume` test (`RESUME_TEST_ARCH`) | 04 and 15 smoke on every smoke model, 24 smoke on those of `MASKING_ARCHS` | ~25 min |
| 10. Smoke C | 22 smoke on a mask smoke model, then 23 (CPU) | | ~10 min |
| 12. Full | 21 for every image, then models (03, 04, 15, 24) | models (mask-free first) | from the estimate |
| 13. ASA | 22 on the mask model | 22 on the baseline | from the estimate |

**Resume.** Kaggle stops a session after 12 hours. Every training writes a backup after each epoch, and the job runner stops every job at `HARD_STOP_H` so the version can still save its outputs. To continue: *Add Input* > *Your Work* > this notebook's stopped version, set `PREVIOUS_OUTPUT` (cell 2) to its folder under `/kaggle/input` and run again with Save & Run All. Cell 8 copies the earlier models, masks and results and checks that the split is identical. The smoke runs and checks of cells 9-11 run again (about 1 h), then finished steps are skipped (a 03 step only when its weights and test results are written too), every model continues from its last finished epoch, 24 runs again if it had not finished (it has no `--resume`), and 22 continues after its last finished batch (`--resume`: it writes its rows after every batch). 22 starts when its smoke run and one batch fit before `HARD_STOP_H`.

## 1-3. Setup, settings and the commands of the session

In [ ]:
# 1. Setup: clone the repo at BRANCH, install the pinned tf-keras, record the commit
import os, subprocess, time
SESSION_START = time.time()   # Kaggle ends a session after 12 h; the job runner (cell 4) watches this clock

BRANCH = 'main'   # must hold the anatomy build (03 --arch/--masks, scripts 21-24, configs/anatomy.yml); cell 6 checks

!pip install -q --no-deps tf-keras==2.20.1

from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret('github_token')
REPO = '/tmp/ocular-adv-xai'
url = f'https://{tok}@github.com/nandininautiyal/ocular-adv-xai.git'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
    !git -C {REPO} remote set-url origin https://github.com/nandininautiyal/ocular-adv-xai.git
else:
    # a reused session may have another branch checked out: always land exactly on BRANCH
    !git -C {REPO} fetch -q {url} {BRANCH}
    !git -C {REPO} checkout -q -f -B {BRANCH} FETCH_HEAD
%cd {REPO}
!git log -3 --oneline
COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('branch:', BRANCH, '| commit:', COMMIT)
!nvidia-smi -L

In [ ]:
# 2. Settings: the only cell to edit
RUN_FULL = False      # False: checks, data, masks, smoke runs and the time estimate. True: also the full runs
                      # (use Save Version > Save & Run All)
ARCHS = ['attn_mask', 'attn', 'gap']   # 03 --arch; the minimum set for the BTP (the baseline comes from
                                       # train_cross_session.ipynb via BASELINE_CKPT). Later: 'parts',
                                       # 'attnet_gate'; add 'baseline' to train the vendored model here as well
SEEDS = [1]           # one model per arch and seed
MAX_FRAME = 1009      # 02 and 15 --max-frame: leave out the frames numbered 1010 and up (16 found the
                      # enrollment ones synthetic); None keeps every frame
CFG = 'configs/anatomy.yml'   # configs/cross_session.yml (same recipe as the cross-session baseline, epoch cap 40
                              # instead of 30) + anatomy section
TAG = 'an'            # full models are tagged an_<arch>_s<seed>: Xception_ArcFace_an_parts_s1.keras, run_an_parts_s1.json
SMOKE = 'smoke'       # tag prefix of the smoke runs (smoke_<arch>_s<seed>, smoke_resume, ...)
SMOKE_CLASSES = 20    # classes of the smoke tree (02 --max-classes)
SMOKE_EPOCHS = 2      # epochs of each smoke training; the last one times a steady epoch
RESUME_TEST_ARCH = 'attn_mask'   # the --resume test trains this arch (the mask-loss trainer) and stops it after epoch 1
RESUME_TEST_EPOCHS = 3
MASK_LOSS_WEIGHT = None      # None: the config's anatomy.mask_loss_weight (1.0); else 03 --mask-loss-weight
BASELINE_CKPT = None  # 22's comparison model: the cross-session baseline of train_cross_session.ipynb, e.g.
                      # '/kaggle/input/<that notebook>/trained_models/Xception_ArcFace_cs_s1.keras' (its
                      # class_index_cs_s1.json next to it). None: the 'baseline' arch if in ARCHS, else no comparison
ASA_ARCH = 'auto'     # the mask model 22 explains: 'auto' = lowest P2 EER of attn_mask / parts (seed SEEDS[0]), or an arch
ASA_POOL_PER_CLASS = 4       # verification images per class that 17 segments at 256 x 256: the images 22 can draw
ASA_MAX_IMAGES = None        # images of each full 22 run (one per class; None = every class in the pool)
ASA_SMOKE_IMAGES = 4
ASA_WORKERS = max(1, (os.cpu_count() or 2) // 2)   # inpainting threads of each 22 run: two runs share the CPUs
ASA_PAIR_FACTOR = 1.5        # allowance for the CPU work two parallel 22 runs still share (TensorFlow, defences);
                             # a guess, not a measurement: 22 resumes, so a low estimate costs a stop, not the run
ASA_ATTACKS = 'fgsm,pgd,aas_pgd'     # 22 --attacks; aas_pgd = PGD through the attention-temperature-scaled model
                                     # (left out on models without attention: pgd itself there, cell 3); add
                                     # bpda_pgd for BPDA through each
                                     # defence (12 more condition images per image)
ASA_VERIF_PAIRS = True       # 22 --verif-pairs: also explain the verification score against a genuine and an impostor
                             # template (the pool holds verification images only, so the genuine template is another
                             # verification image of the subject, not one of the other session)
ASA_BASELINE_HIDING = 'blur,mean'    # 22 --baseline-hiding of the KernelSHAP / LIME part baselines (None: all hiding
                                     # methods; inpaint about doubles their CPU time)
ASA_PIXEL_CONDITIONS = 'clean'       # 22 --pixel-conditions of the pixel baselines (each: 2 x 500 forward passes per image)
MASKING_ARCHS = ['attn_mask', 'gap']   # 24, the gradient-masking check (smoke and full, seed SEEDS[0]); a
                                                # robustness claim needs it without flags; not for attnet_gate
MASKING_SMOKE_IMAGES = 16    # images of each 24 smoke run: one full batch (24's --batch-size 16), as a batch of 4
                             # costs about as much as one of 16; the full runs take one image per class
MASKING_MAX_IMAGES = 100     # images of each full 24 run: one per class, at most this many (None = every class).
                             # 24 decides by success rates near 0 or 1, not by per-subject statistics: at 100
                             # images a rate of 1.0 has a 95 % Wilson interval of [0.963, 1]. 300 images cost
                             # about 3 h per model on Kaggle's T4 (22 Nov), 100 about 1 h
N_BOOT = 1000         # bootstrap resamples of 15 and 23 (the smoke runs use 100)
PREVIOUS_OUTPUT = None       # resume: the folder under /kaggle/input that holds an earlier version's output
START_IF_TOO_LONG = False    # True: start the trainings even if the estimate ends after HARD_STOP_H; they stop there
                             # with a backup of the last finished epoch, and the next session resumes them
HARD_STOP_H = 11.25   # any job still running at this point is stopped, so that the version can still save its outputs
ZIP_MAX_MB = 100      # files larger than this stay out of results.zip (checkpoints, masks and images always do)

W = '/kaggle/working'
LOGS = f'{W}/logs'
N_CLASSES = 300                                         # VISOB IDs 1141-3279 plus 7000
MODELS, SMOKE_MODELS = f'{W}/trained_models', f'{W}/smoke_models'
CS_TREE, CS_MANIFEST = '/tmp/ocular_cross_session', f'{W}/manifest_split_cross_session.csv'   # 02's defaults
SMALL_TREE, SMALL_MANIFEST = '/tmp/ocular_cs_small', f'{W}/manifest_split_cs_small.csv'
OPEN_MANIFEST = f'{W}/manifest_open_set.csv'
SPLITS = f'{W}/splits'
OPEN_SET = f'{SPLITS}/open_set_test.csv'
EVAL, VERIF = f'{W}/eval_clean', f'{W}/verification'
EYE_MASKS, TRAIN_MASKS = f'{W}/eye_masks', f'{W}/train_masks'
ASA, ANALYSIS = f'{W}/asa', f'{W}/analysis_asa'
MASKING = f'{W}/masking'

In [ ]:
# 3. Every command of the session, in one place (nothing runs in this cell)
MF = f' --max-frame {MAX_FRAME}' if MAX_FRAME is not None else ''   # 02 (cross_session, open_set) and 15
MASK_ARCHS = ('attn_mask', 'parts', 'attnet_gate')      # 03 needs --masks (src/models/anatomy_model.MASK_ARCHS)
LOSS_ARCHS = ('attn_mask', 'parts')                     # trained with the mask loss
TEST_MASK_ARCHS = ('attnet_gate',)                      # need the masks at test time too (04 / 15 --region-masks)
MASKS64 = f'{TRAIN_MASKS}/masks64_{TAG}.npz'            # 21: every image of the tree (+ the open set for attnet_gate)
SMALL_MASKS64 = f'{TRAIN_MASKS}/masks64_{SMOKE}.npz'    # 21: the smoke tree
POOL_MASKS = f'{EYE_MASKS}/masks_asa_pool.npz'          # 17: the ASA pool, 256 x 256 (22 --masks)
MASK_LIST, ASA_POOL = f'{W}/mask_list.csv', f'{W}/asa_pool.csv'                      # written in cell 7
ASA_LIST, ASA_SMOKE_LIST = f'{W}/asa_images.csv', f'{W}/asa_smoke_images.csv'       # written in cells 13 and 10

def tag(arch, seed, prefix=TAG):
    return f'{prefix}_{arch}_s{seed}'

def ckpt(models, t):
    return f'{models}/Xception_ArcFace_{t}.keras'

def class_index(path):
    # class_index_<tag>.json next to a 03 checkpoint Xception_ArcFace_<tag>.keras
    folder, name = os.path.split(path)
    return f"{folder}/class_index_{name[len('Xception_ArcFace_'):-len('.keras')]}.json"

train03 = f'python -u scripts/03_train_baseline.py --cfg {CFG} --resume --reshuffle --save-weights --verbose 2'
def train_cmd(arch, seed, t, split_dir, n_classes, out_dir, masks, epochs=None):
    cmd = f'{train03} --arch {arch} --split-dir {split_dir} --n-classes {n_classes} --seed {seed} --out-dir {out_dir} --tag {t}'
    if arch in MASK_ARCHS:
        cmd += f' --masks {masks}'
    if MASK_LOSS_WEIGHT is not None and arch in LOSS_ARCHS:
        cmd += f' --mask-loss-weight {MASK_LOSS_WEIGHT}'
    return cmd + (f' --epochs {epochs}' if epochs else '')

def region(arch, masks):
    return f' --region-masks {masks}' if arch in TEST_MASK_ARCHS else ''

eval04 = 'python -u scripts/04_eval_clean.py'
def eval_cmd(path, t, split_dir, n_classes, manifest, masks='', arch=''):
    return (f'{eval04} --split-dir {split_dir} --n-classes {n_classes} --ckpt {path} --class-index {class_index(path)} '
            f'--manifest {manifest} --out-dir {EVAL} --tag {t}' + region(arch, masks))

verif15 = f'python -u scripts/15_eval_verification.py{MF} --manifest {W}/manifest_raw.csv'
def verif_cmd(path, t, masks, arch, smoke=False):
    modes = 'p2' if smoke and arch in TEST_MASK_ARCHS else 'p2,p3'     # the smoke masks cover no open-set image
    extra = ' --max-subjects 10 --n-boot 100' if smoke else f' --n-boot {N_BOOT}'
    return (f'{verif15} --modes {modes} --p3-list {OPEN_SET} --ckpt {path} --class-index {class_index(path)}'
            f'{extra} --out-dir {VERIF} --tag {t}' + region(arch, masks))

asa22 = 'python -u scripts/22_anatomical_shapley.py'
analyse23 = 'python -u scripts/23_analyse_asa.py'
# the same 22 flags in every 22 run of the session, so the smoke runs time the full runs; aas_pgd only on models
# with an attention pool: on any other model it is pgd itself, so 22 would run every pgd condition twice and 23
# would count the copies as separate tests
ATTENTION_ARCHS = ('attn', 'attn_mask', 'parts')
def asa_flags(arch):
    attacks = ','.join(a for a in ASA_ATTACKS.split(',') if a != 'aas_pgd' or arch in ATTENTION_ARCHS)
    return (f' --attacks {attacks} --pixel-conditions {ASA_PIXEL_CONDITIONS}'
            + (f' --baseline-hiding {ASA_BASELINE_HIDING}' if ASA_BASELINE_HIDING else '')
            + (' --verif-pairs' if ASA_VERIF_PAIRS else ''))
def asa_cmds(name, path, smoke_list=ASA_LIST):
    # 22 smoke and full run on one model (same image list and seeds for every model, so the images are paired),
    # then 23 on each; the full run leaves out the smoke images and continues a stopped run (--resume). The two
    # models run in parallel, so each gets half the CPU threads for inpainting (--workers). 'base' is the
    # Xception baseline (no attention pool)
    flags = asa_flags(ASA_MASK_ARCH if name == 'mask' else 'baseline')
    return {f'22_smoke_{name}': f'{asa22} --ckpt {path} --correct-csv {smoke_list} --masks {POOL_MASKS} '
                                f'--max-images {ASA_SMOKE_IMAGES} --seed 1 --workers {ASA_WORKERS} --out-dir {ASA} '
                                f'--tag smoke_{name}{flags}',
            f'22_{name}': f'{asa22} --ckpt {path} --correct-csv {ASA_LIST} --masks {POOL_MASKS} --per-class 1 '
                          f'--seed 2 --exclude-from {ASA}/images_asa_smoke_{name}.csv --workers {ASA_WORKERS} --resume '
                          f'--out-dir {ASA} --tag {TAG}_{name}{flags}'
                          + (f' --max-images {ASA_MAX_IMAGES}' if ASA_MAX_IMAGES else ''),
            f'23_{name}': f'{analyse23} --rows {ASA}/rows_asa_{TAG}_{name}.csv --n-boot {N_BOOT} '
                          f'--out-dir {ANALYSIS}/{TAG}_{name}'}

mask24 = 'python -u scripts/24_gradient_masking.py'
MASK_CHECK = [a for a in MASKING_ARCHS if a in ARCHS]   # the archs of this session that get the masking check
def masking_cmd(path, t, smoke=False):
    # 24 with its default grids on the model's own correct-image list (04); the smoke run times the full run
    sample = (f'--max-images {MASKING_SMOKE_IMAGES} --seed 1' if smoke else '--per-class 1 --seed 2'
              + (f' --max-images {MASKING_MAX_IMAGES}' if MASKING_MAX_IMAGES else ''))
    return f'{mask24} --ckpt {path} --correct-csv {EVAL}/correct_test_{t}.csv {sample} --out-dir {MASKING} --tag {t}'

CMD = {
    # data, in the foreground (cell 7)
    '01_manifest':      f'python -u scripts/01_build_manifest.py --out {W}/manifest_raw.csv',
    '02_cross_session': f'python -u scripts/02_make_splits.py --protocol cross_session{MF} --write-split-csvs {SPLITS}',
    '02_open_set':      f'python -u scripts/02_make_splits.py --protocol open_set{MF} --write-split-csvs {SPLITS}',
    '02_small':         f'python -u scripts/02_make_splits.py --protocol cross_session{MF} --max-classes {SMOKE_CLASSES} '
                        f'--out-dir {SMALL_TREE} --out-manifest {SMALL_MANIFEST}',
    # masks (smoke A and the full run)
    '21_smoke':         f'python -u scripts/21_make_train_masks.py --split-dir {SMALL_TREE} --splits train,validation,test '
                        f'--cfg {CFG} --out-dir {TRAIN_MASKS} --tag {SMOKE}',
    '17_pool':          f'python -u scripts/17_make_eye_masks.py --csv {ASA_POOL} --out-dir {EYE_MASKS} --tag asa_pool',
    '21_full':          f'python -u scripts/21_make_train_masks.py --csv {MASK_LIST} '
                        f'--splits train,validation,test,open_set --cfg {CFG} --out-dir {TRAIN_MASKS} --tag {TAG}',
}
SMOKE_TAGS = {a: tag(a, SEEDS[0], SMOKE) for a in ARCHS}
for a, st in SMOKE_TAGS.items():          # smoke A (cell 9): every arch for SMOKE_EPOCHS epochs on the small tree
    CMD[f'03_{st}'] = train_cmd(a, SEEDS[0], st, SMALL_TREE, SMOKE_CLASSES, SMOKE_MODELS, SMALL_MASKS64, SMOKE_EPOCHS)
# smoke B (cell 10): the same command twice, the first run stopped after its first epoch; 04 and 15 on every smoke model
CMD['03_resume_a'] = CMD['03_resume_b'] = train_cmd(RESUME_TEST_ARCH, SEEDS[0], f'{SMOKE}_resume', SMALL_TREE,
                                                    SMOKE_CLASSES, SMOKE_MODELS, SMALL_MASKS64, RESUME_TEST_EPOCHS)
for a, st in SMOKE_TAGS.items():
    CMD[f'04_{st}'] = eval_cmd(ckpt(SMOKE_MODELS, st), st, SMALL_TREE, SMOKE_CLASSES, SMALL_MANIFEST, SMALL_MASKS64, a)
    CMD[f'15_{st}'] = verif_cmd(ckpt(SMOKE_MODELS, st), st, SMALL_MASKS64, a, smoke=True)
    if a in MASK_CHECK:
        CMD[f'24_{st}'] = masking_cmd(ckpt(SMOKE_MODELS, st), st, smoke=True)
# smoke C (cell 10): 22 + 23 on the smoke model of a mask arch (parts if trained: it also has the receipt)
SMOKE_ASA_ARCH = next((a for a in ('parts', 'attn_mask', 'attn') if a in ARCHS), ARCHS[0])
CMD['22_smoke'] = (f'{asa22} --ckpt {ckpt(SMOKE_MODELS, SMOKE_TAGS[SMOKE_ASA_ARCH])} --correct-csv {ASA_SMOKE_LIST} '
                   f'--masks {POOL_MASKS} --max-images {ASA_SMOKE_IMAGES} --seed 1 --workers {ASA_WORKERS} '
                   f'--out-dir {ASA} --tag {SMOKE}{asa_flags(SMOKE_ASA_ARCH)}')     # a full run's threads and flags, to time it
CMD['23_smoke'] = f'{analyse23} --rows {ASA}/rows_asa_{SMOKE}.csv --n-boot 100 --out-dir {ANALYSIS}/{SMOKE}'
# full (cell 12): per model 03 -> 04 -> 15 P2 + P3 (-> 24 for the archs of MASK_CHECK, seed SEEDS[0]) on one GPU
FULL_TAGS = {(a, s): tag(a, s) for a in ARCHS for s in SEEDS}
for (a, s), t in FULL_TAGS.items():
    CMD[f'03_{t}'] = train_cmd(a, s, t, CS_TREE, N_CLASSES, MODELS, MASKS64)
    CMD[f'04_{t}'] = eval_cmd(ckpt(MODELS, t), t, CS_TREE, N_CLASSES, CS_MANIFEST, MASKS64, a)
    CMD[f'15_{t}'] = verif_cmd(ckpt(MODELS, t), t, MASKS64, a)
    if a in MASK_CHECK and s == SEEDS[0]:
        CMD[f'24_{t}'] = masking_cmd(ckpt(MODELS, t), t)
MODEL_STEPS = ('03', '04', '15', '24')       # the steps of one model, in order (24 only where it is in CMD)
# ASA (cell 13): the mask model and the baseline; ASA_ARCH = 'auto' is resolved there (parts until then)
ASA_MASK_ARCH = ASA_ARCH if ASA_ARCH != 'auto' else ('parts' if 'parts' in ARCHS else next(
    (a for a in LOSS_ARCHS if a in ARCHS), ARCHS[0]))
BASE = BASELINE_CKPT or (ckpt(MODELS, tag('baseline', SEEDS[0])) if 'baseline' in ARCHS else None)
if BASELINE_CKPT:                  # its correct-image list, with its own class index
    CMD['04_base_ext'] = eval_cmd(BASELINE_CKPT, 'base_ext', CS_TREE, N_CLASSES, CS_MANIFEST)
CMD.update(asa_cmds('mask', ckpt(MODELS, tag(ASA_MASK_ARCH, SEEDS[0]))))
if BASE:
    CMD.update(asa_cmds('base', BASE))
# a background step runs only after these steps succeeded (it waits for steps in the other GPU's chain)
NEEDS = {'03_resume_a': ['21_smoke'], '03_resume_b': ['21_smoke'],
         **{f'03_{st}': ['21_smoke'] for a, st in SMOKE_TAGS.items() if a in MASK_ARCHS},
         **{f'{step}_{st}': [f'03_{st}'] for st in SMOKE_TAGS.values() for step in ('04', '15')},
         **{f'24_{st}': [f'04_{st}'] for st in SMOKE_TAGS.values() if f'24_{st}' in CMD},
         **{f'03_{t}': ['21_full'] for (a, s), t in FULL_TAGS.items() if a in MASK_ARCHS},
         **{f'{step}_{t}': [f'03_{t}'] for t in FULL_TAGS.values() for step in ('04', '15')},
         **{f'24_{t}': [f'04_{t}'] for t in FULL_TAGS.values() if f'24_{t}' in CMD},
         **{f'22_{n}': [f'22_smoke_{n}'] for n in ('mask', 'base') if f'22_{n}' in CMD}}
for name, cmd in CMD.items():
    print(f'{name:28} {cmd}')

## 4-5. Helpers and checks

The job runner starts one chain of steps per GPU in the background. Each step writes `logs/<step>.log`, `logs/<step>.start` and `logs/<step>.exit`; a step that needs a step of the other chain (e.g. a mask model and the masks of 21) waits for it and runs only if it succeeded. Every job is stopped at `HARD_STOP_H` (after waiting, at most 4 min, until no checkpoint is being written). `plan_chains` spreads the trainings over the two GPUs by their estimated hours, the mask models after 21 on the first GPU.

In [ ]:
# 4. Helpers: foreground commands, background jobs (one GPU each), the 12-hour guard, step and epoch timings
import glob, hashlib, json, re, shutil, signal
import numpy as np
import pandas as pd
os.makedirs(LOGS, exist_ok=True)
FG = {}      # exit status of the foreground steps run with sh(..., check=False)

def hours():
    return (time.time() - SESSION_START) / 3600

def sh(cmd, check=True, gpu=None):
    # run cmd from the repo with its output shown here; check: stop the notebook if it fails
    print('$', cmd, flush=True)
    env = dict(os.environ, **({'CUDA_VISIBLE_DEVICES': str(gpu)} if gpu is not None else {}))
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, executable='/bin/bash', text=True, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    code = p.wait()
    assert code == 0 or not check, f'exit code {code}: {cmd}'
    return code

def py(snippet):
    # (exit code, last output line) of a Python snippet run in a fresh process (keeps TensorFlow out of the notebook)
    r = subprocess.run(['python', '-c', snippet], cwd=REPO, capture_output=True, text=True)
    out = r.stdout.strip().splitlines() or r.stderr.strip().splitlines() or ['']
    return r.returncode, out[-1]

def load(path):
    with open(path) as f:
        return json.load(f)

def tail(path, n=12):
    try:
        with open(path, errors='ignore') as f:
            lines = [line.rstrip() for line in f if line.strip()]
    except FileNotFoundError:
        return ['(no log)']
    return lines[-n:] or ['(empty log)']

def status(name):
    # 0 ok, another number failed, 'skipped' (a step it needs failed), None (not run, or stopped)
    try:
        text = open(f'{LOGS}/{name}.exit').read().strip()
    except FileNotFoundError:
        return None
    return int(text) if text.lstrip('-').isdigit() else text

def seconds(name):
    # wall-clock seconds of a finished background step (from its .start file to its .exit file), else None
    try:
        start = float(open(f'{LOGS}/{name}.start').read())
        return os.path.getmtime(f'{LOGS}/{name}.exit') - start
    except (FileNotFoundError, ValueError):
        return None

EPOCH_LINE = re.compile(r'^\s*\d+/\d+ - (\d+)s - .*val_loss')
def epoch_seconds(name):
    # seconds of every epoch in a 03 log (Keras --verbose 2 prints one 'steps/steps - 95s - loss: ... - val_loss: ...'
    # line per epoch; the mask-loss models add ce / mask_loss to it)
    try:
        with open(f'{LOGS}/{name}.log', errors='ignore') as f:
            return [int(m.group(1)) for m in map(EPOCH_LINE.match, f) if m]
    except FileNotFoundError:
        return []

def csv_rows(path):
    # data rows of a CSV that may still be written (CSVLogger creates it empty at the start of training)
    try:
        with open(path, errors='ignore') as f:
            return max(sum(1 for line in f if line.strip()) - 1, 0)
    except FileNotFoundError:
        return 0

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(2**24), b''):
            h.update(block)
    return h.hexdigest()

def mtime(path):
    try:
        return os.path.getmtime(path)
    except OSError:
        return 0

def backup_written(folder):
    # True once tf.keras BackupAndRestore has finished a backup in folder (the 'checkpoint' state file comes last)
    return bool(glob.glob(f'{folder}/**/checkpoint', recursive=True))

def settle(folders, quiet=60, limit=240, every=5):
    # wait (at most limit s) until no file in folders has been written for quiet s, so that a stop does not cut a
    # checkpoint in half
    t0 = time.time()
    while time.time() - t0 < limit:
        files = [p for d in folders for p in glob.glob(f'{d}/**', recursive=True) if os.path.isfile(p)]
        if time.time() - max([mtime(p) for p in files] or [0]) > quiet:
            return time.time() - t0
        time.sleep(every)
    print(f'files still being written after {limit} s; stopping anyway')
    return time.time() - t0

def clear(names):
    # remove the log, exit and start files of steps about to run (a stale .exit would satisfy a waiting step)
    for name in names:
        for ext in ('log', 'exit', 'start'):
            if os.path.exists(f'{LOGS}/{name}.{ext}'):
                os.remove(f'{LOGS}/{name}.{ext}')

def launch(label, names, gpu=None, cleared=False):
    # run the steps CMD[name] one after the other in a background job on one GPU (gpu=None: CPU only); a step first
    # waits for the steps it needs (they may run in another job) and runs only if they succeeded
    lines = [f'cd {REPO}', f"export CUDA_VISIBLE_DEVICES={'' if gpu is None else gpu}",
             'export PYTHONUNBUFFERED=1 TF_CPP_MIN_LOG_LEVEL=1']
    if not cleared:
        clear(names)
    for name in names:
        needs = NEEDS.get(name, [])
        wait_for = ''.join(f'while [ ! -f {LOGS}/{n}.exit ]; do sleep 20; done; ' for n in needs)
        ok = ' && '.join(f'[ "$(cat {LOGS}/{n}.exit 2>/dev/null)" = 0 ]' for n in needs) or 'true'
        lines.append(f'{wait_for}if {ok}; then date +%s > {LOGS}/{name}.start; {CMD[name]} > {LOGS}/{name}.log 2>&1; '
                     f'echo $? > {LOGS}/{name}.exit; '
                     f'else echo "skipped: needs {" ".join(needs)}" > {LOGS}/{name}.log; '
                     f'echo skipped > {LOGS}/{name}.exit; fi')
    proc = subprocess.Popen(['bash', '-c', '\n'.join(lines)], start_new_session=True)
    print(f'{label}: {" -> ".join(names)}', flush=True)
    return {'label': label, 'names': names, 'proc': proc}

def launch_all(chains):
    # chains: {gpu: [step, ...]}; clears every step first, so no job sees a stale .exit of another job's step
    clear([n for names in chains.values() for n in names])
    return [launch(f'GPU {g}', names, gpu=g, cleared=True) for g, names in chains.items() if names]

def stop(jobs):
    # stop the running jobs with all their processes (each job is its own process group): SIGTERM, then SIGKILL
    running = [j for j in jobs if j['proc'].poll() is None]
    for sig in (signal.SIGTERM, signal.SIGKILL):
        for job in running:
            try:
                os.killpg(job['proc'].pid, sig)
            except ProcessLookupError:
                pass
        if sig == signal.SIGTERM:
            time.sleep(30)

def wait(jobs, every=60, also=()):
    # wait for jobs, printing the last log line of each running step; stop every job at HARD_STOP_H;
    # print and return {step: status}
    while any(j['proc'].poll() is None for j in jobs):
        time.sleep(every)
        if hours() > HARD_STOP_H:
            print(f'{hours():.2f} h: past HARD_STOP_H = {HARD_STOP_H}, stopping every job so the outputs can be saved '
                  f'(first waiting until no checkpoint is being written)', flush=True)
            settle([MODELS, SMOKE_MODELS])
            stop(list(jobs) + list(also))
            break
        parts = []
        for j in list(jobs) + list(also):
            step = next((n for n in j['names'] if status(n) is None), None) if j['proc'].poll() is None else None
            parts.append(f"{j['label']} {step}: {tail(f'{LOGS}/{step}.log', 1)[0][:150]}" if step
                         else f"{j['label']}: done")
        print(time.strftime('%H:%M'), f'({hours():.1f} h)', ' | '.join(parts), flush=True)
    codes = {n: status(n) for j in jobs for n in j['names']}
    for name, code in codes.items():
        word = {0: 'ok', None: 'NOT RUN (or stopped)', 'skipped': 'SKIPPED (a step it needs failed)'}.get(
            code, f'FAILED (exit {code})')
        print(f'\n---- {name}: {word}')
        print('\n'.join(tail(f'{LOGS}/{name}.log', 12 if code == 0 else 40)))
    return codes

def plan_chains(jobs, first=(), first_h=0.0, n_gpu=2):
    # jobs: [(arch, step names, hours)]. GPU 0 runs `first` (21, first_h hours) before anything else; mask models
    # (they wait for 21) start no earlier than first_h, so each GPU runs its mask-free models first. Every split of
    # the jobs over the GPUs is tried (at most 12 jobs; more fall back to giving each job, longest first, to the
    # GPU where it would finish sooner) and the one that ends soonest is kept: a greedy split in a fixed order can
    # leave one GPU hours longer than needed. Returns ({gpu: [steps]}, {gpu: hours})
    import itertools
    def lay_out(assignment):
        chains = {g: list(first) if g == 0 else [] for g in range(n_gpu)}
        load = {g: first_h if g == 0 else 0.0 for g in range(n_gpu)}
        for (arch, steps, h), g in sorted(zip(jobs, assignment), key=lambda x: x[0][0] in MASK_ARCHS):
            chains[g] += steps
            load[g] = max(load[g], first_h if arch in MASK_ARCHS else 0.0) + h
        return chains, load
    if len(jobs) <= 12:
        best = min(itertools.product(range(n_gpu), repeat=len(jobs)),
                   key=lambda a: (max(lay_out(a)[1].values()), a))
        return lay_out(best)
    load = {g: first_h if g == 0 else 0.0 for g in range(n_gpu)}
    assignment = [0] * len(jobs)
    for i in sorted(range(len(jobs)), key=lambda i: -jobs[i][2]):
        arch, _, h = jobs[i]
        ready = first_h if arch in MASK_ARCHS else 0.0
        g = min(load, key=lambda k: max(load[k], ready) + h)
        assignment[i], load[g] = g, max(load[g], ready) + h
    return lay_out(assignment)

def with_templates(frame):
    # 22 --verif-pairs stops if a sampled image has no other image of its class in --correct-csv (its genuine
    # template): with ASA_VERIF_PAIRS, keep only the classes with at least two images
    if not ASA_VERIF_PAIRS:
        return frame
    return frame[frame.groupby('true_idx').relpath.transform('size') >= 2]

def zip_keep(rel):
    # results.zip keeps records, logs, tables and plots: no checkpoints, no masks, no images (17's QC overlays)
    name = os.path.basename(rel)
    if name.endswith(('.keras', '.h5', '.npz', '.pth')) or '/backup_' in f'/{rel}':
        return False
    return not (rel.startswith('eye_masks/') and name.endswith(('.png', '.jpg')))

In [ ]:
# 5. Checks of each step's outputs; each returns [(check, passed, detail, hard)], hard = must pass
RUN_KEYS = ['finished', 'git_commit', 'git_dirty', 'argv', 'args', 'cfg', 'versions', 'split_dir', 'images_per_split',
            'images_by_phase', 'class_list_sha256', 'epochs_run', 'best_epoch', 'best_val_loss', 'stopped_early',
            'minutes', 'resumed', 'checkpoint', 'checkpoint_sha256', 'weights', 'weights_sha256']
ANATOMY_KEYS = ['arch', 'anatomy', 'masks', 'masks_sha256', 'mask_classes', 'mask_coverage']   # 03 --arch other
                                                                                                # than baseline
def same_commit(value):
    return isinstance(value, str) and len(value) >= 7 and COMMIT.startswith(value)

def split_counts(manifest):
    return pd.read_csv(manifest).groupby(['split', 'phase']).size().unstack(fill_value=0)

def checks_split(manifest, n_classes):
    # 02 --protocol cross_session: test = verification images, train/validation = enrollment images, whole buckets
    m = pd.read_csv(manifest)
    name = os.path.basename(manifest)
    phases = {s: set(m.phase[m.split == s]) for s in ('train', 'validation', 'test')}
    bucket = ['subject', 'condition', 'eye']
    shared = (set(map(tuple, m[m.split == 'train'][bucket].values))
              & set(map(tuple, m[m.split == 'validation'][bucket].values)))
    return [(f'02 {name}: test = verification images only', phases['test'] == {'verification'},
             str(sorted(phases['test'])), True),
            (f'02 {name}: train and validation = enrollment images only',
             phases['train'] == phases['validation'] == {'enrollment'}, '', True),
            (f'02 {name}: {n_classes} classes in train', m.subject[m.split == 'train'].nunique() == n_classes,
             f"{m.subject[m.split == 'train'].nunique()} classes", True),
            (f'02 {name}: no (subject, condition, eye) bucket in both train and validation', not shared,
             f'{len(shared)} shared', True)]

def checks_masks(t, wanted):
    # 21: one 64 x 64 map per image of the wanted splits, the part-presence report, the commit
    path = f'{TRAIN_MASKS}/masks64_summary_{t}.json'
    if not os.path.exists(path):
        return [(f'21 {t}: summary', False, f'{path} missing', True)]
    s = load(path)
    per = s.get('images_per_split') or {}
    out = [(f'21 {t}: masks for every image of {sorted(wanted)}',
            all(per.get(k, 0) == n for k, n in wanted.items()), f'{per} (wanted {wanted})', True),
           (f'21 {t}: masks64 file and part-presence report written',
            os.path.exists(f'{TRAIN_MASKS}/masks64_{t}.npz') and os.path.exists(f'{TRAIN_MASKS}/part_presence_{t}.csv'),
            '', True),
           (f'21 {t}: commit and clean checkout', same_commit(s.get('code_commit')) and s.get('code_dirty') is False,
            f"{s.get('code_commit')}, dirty {s.get('code_dirty')}", True)]
    if os.path.exists(f'{TRAIN_MASKS}/part_presence_{t}.csv'):
        p = pd.read_csv(f'{TRAIN_MASKS}/part_presence_{t}.csv')
        brow = p[p.group == 'brow']
        out.append((f'21 {t}: eyebrow presence reported', len(brow) > 0,
                    ', '.join(f"{r['split']} {r['share_ge_0.5pct']:.1%}" for _, r in brow.iterrows()), False))
    return out

def checks_03(t, arch, models, n_classes, max_epochs, seed, masks):
    files = {k: f'{models}/{stem}_{t}.{ext}' for k, (stem, ext) in {
        'ckpt': ('Xception_ArcFace', 'keras'), 'weights': ('Xception_ArcFace', 'weights.h5'),
        'class_index': ('class_index', 'json'), 'log': ('log', 'csv'), 'run': ('run', 'json')}.items()}
    missing = [os.path.basename(p) for p in files.values() if not os.path.exists(p)]
    out = [(f'03 {t}: checkpoint, weights, class index, log and run record', not missing, ', '.join(missing), True)]
    if missing:
        return out
    run, idx = load(files['run']), load(files['class_index'])
    res = load(f'{models}/results_{t}.json') if os.path.exists(f'{models}/results_{t}.json') else {}
    log = pd.read_csv(files['log'])
    keys = RUN_KEYS + (ANATOMY_KEYS if arch != 'baseline' else [])
    absent = [k for k in keys if k not in run]
    out.append((f'03 {t}: run record has every field, training finished', not absent and run.get('finished') is True,
                ', '.join(absent) or f"finished {run.get('finished')}", True))
    if arch != 'baseline':
        out.append((f'03 {t}: arch {arch} recorded', run.get('arch') == arch, str(run.get('arch')), True))
    if arch in MASK_ARCHS:
        cov = run.get('mask_coverage') or {}
        out.append((f'03 {t}: masks file recorded (sha256) and a mask for every train/validation/test image',
                    run.get('masks_sha256') == sha256(masks) and all(c.get('share') == 1.0 for c in cov.values()),
                    str(cov), True))
    names = sorted(idx, key=idx.get)
    sha = hashlib.sha256('\n'.join(names).encode('utf-8')).hexdigest()
    out.append((f'03 {t}: class index of {n_classes} classes, sha256 recorded',
                len(idx) == n_classes and run.get('class_list_sha256') == sha, f'{len(idx)} classes', True))
    out.append((f'03 {t}: commit and clean checkout', same_commit(run.get('git_commit')) and run.get('git_dirty') is False,
                f"{run.get('git_commit')}, dirty {run.get('git_dirty')}", True))
    args = run.get('args') or {}
    out.append((f'03 {t}: seed {seed}, --resume, --reshuffle and --save-weights recorded',
                args.get('seed') == seed and all(args.get(k) is True for k in ('resume', 'reshuffle', 'save_weights')),
                str({k: args.get(k) for k in ('seed', 'resume', 'reshuffle', 'save_weights', 'cfg')}), True))
    epochs = log.epoch.tolist() if 'epoch' in log else []
    out.append((f'03 {t}: at most {max_epochs} epochs, each logged once', 0 < len(epochs) <= max_epochs
                and epochs == list(range(len(epochs))), f'log epochs {epochs[:3]}...{epochs[-2:]}', True))
    if arch in LOSS_ARCHS:
        out.append((f'03 {t}: CE and mask loss logged separately (train and validation)',
                    {'ce', 'mask_loss', 'val_ce', 'val_mask_loss'} <= set(log.columns), ', '.join(log.columns), True))
    out.append((f'03 {t}: run record matches the checkpoint and the weights file (sha256)',
                run.get('checkpoint_sha256') == sha256(files['ckpt'])
                and run.get('weights_sha256') == sha256(files['weights']), '', True))
    phases = res.get('images_by_phase') or run.get('images_by_phase') or {}
    out.append((f'03 {t}: trained on enrollment images only, tested on verification images only',
                set(phases.get('train', {})) == set(phases.get('validation', {})) == {'enrollment'}
                and set(phases.get('test', {})) == {'verification'}, str(phases), True))
    acc = res.get('test_acc_margin_free')
    out.append((f'03 {t}: test accuracy (margin-free) in results_{t}.json', isinstance(acc, float) and 0 <= acc <= 1,
                f'{acc}', True))
    return out

def checks_resume(rows_at_stop):
    t = f'{SMOKE}_resume'
    if not os.path.exists(f'{SMOKE_MODELS}/run_{t}.json'):
        return [('03 --resume test: run record', False, 'missing', True)]
    run, log = load(f'{SMOKE_MODELS}/run_{t}.json'), pd.read_csv(f'{SMOKE_MODELS}/log_{t}.csv')
    stopped_mid = rows_at_stop is not None and 1 <= rows_at_stop < RESUME_TEST_EPOCHS
    return [(f'03 --resume test ({RESUME_TEST_ARCH}): first run stopped after a finished epoch, before the last',
             stopped_mid, f'{rows_at_stop} epochs logged when it was stopped', True),
            ('03 --resume test: continued (resumed recorded), every epoch logged exactly once',
             run.get('resumed') is True and log.epoch.tolist() == list(range(RESUME_TEST_EPOCHS)),
             f"resumed {run.get('resumed')}, log epochs {log.epoch.tolist()}", True)]

def checks_04(t, manifest, arch=None):
    path = f'{EVAL}/summary_{t}.json'
    if not os.path.exists(path):
        return [(f'04 {t}: summary', False, f'{path} missing', True)]
    s = load(path)
    n_test = int((pd.read_csv(manifest).split == 'test').sum())
    out = [(f'04 {t}: every test image evaluated', s['n_images'] == n_test, f"{s['n_images']} of {n_test}", True)]
    if arch in TEST_MASK_ARCHS:      # 04 records the arch and the region masks of a three-input model
        out.append((f'04 {t}: arch {arch} with --region-masks recorded', s.get('arch') == arch
                    and bool(s.get('region_masks')), f"{s.get('arch')}, {s.get('region_masks')}", True))
    return out

def checks_15(t, modes=('p2', 'p3')):
    mpath, spath = f'{VERIF}/verification_metrics_{t}.csv', f'{VERIF}/verification_summary_{t}.json'
    if not (os.path.exists(mpath) and os.path.exists(spath)):
        return [(f'15 {t}: metrics and summary files', False, f'{mpath} or {spath} missing', True)]
    m, s = pd.read_csv(mpath), load(spath)
    overall = m[m.subset == 'all'].set_index('mode')
    return [(f'15 {t}: {", ".join(modes)} results', set(modes) <= set(overall.index), ', '.join(overall.index), True),
            (f'15 {t}: EER between 0 and 0.5', bool(overall.eer.between(0, 0.5).all()),
             ', '.join(f'{k} EER {r.eer:.4f}' for k, r in overall.iterrows()), True),
            (f'15 {t}: commit', same_commit(s.get('git_commit')), str(s.get('git_commit')), True)]

def checks_asa(t):
    # 22 rows and summary, and 23's report (23 stops on any row-accounting mismatch, so a report means it passed)
    s_path, report_path = f'{ASA}/summary_asa_{t}.json', f'{ANALYSIS}/{t}/report_asa.md'
    if not os.path.exists(s_path):
        return [(f'22 {t}: summary', False, f'{s_path} missing', True)]
    s = load(s_path)
    kinds = s.get('rows_per_target_kind') or {}
    attention = bool((s.get('anatomy') or {}).get('attention'))      # aas_pgd only with an attention pool (cell 3)
    attacks = [a for a in ASA_ATTACKS.split(',') if a != 'aas_pgd' or attention]
    attacks_text = ','.join(attacks)
    return [(f'22 {t}: rows and summary written, commit', os.path.exists(f'{ASA}/rows_asa_{t}.csv')
             and same_commit(s.get('git_commit')), f"{s.get('n_rows')} rows, {s.get('n_images')} images", True),
            (f'22 {t}: attacks {attacks_text}, verification pairs {ASA_VERIF_PAIRS}',
             s.get('attacks') == attacks and (kinds.get('genuine', 0) > 0) == ASA_VERIF_PAIRS,
             f"attacks {s.get('attacks')}, rows per target kind {kinds}", True),
            (f'23 {t}: report written (the row accounting passed)', os.path.exists(report_path), report_path, True)]

def checks_masking(t):
    # 24: rows, summary and commit (hard); a masking flag is a result, not a broken run (WARN), but no robustness
    # claim may be made for a model with flags
    path = f'{MASKING}/summary_masking_{t}.json'
    if not os.path.exists(path):
        return [(f'24 {t}: summary', False, f'{path} missing', True)]
    s = load(path)
    flags, skipped = s.get('masking_flags'), s.get('checks_skipped') or []
    return [(f'24 {t}: rows and summary written, commit', os.path.exists(f'{MASKING}/rows_masking_{t}.csv')
             and same_commit(s.get('git_commit')) and isinstance(flags, list),
             f"{s.get('n_images')} images, {s.get('arch')}, attention pool {s.get('has_attention')}", True),
            (f'24 {t}: no sign of gradient masking (else no robustness claim for this model)', flags == [],
             f"flags: {', '.join(flags or []) or 'none'} | skipped: {', '.join(skipped) or 'none'}", False)]

def run_checks(*specs):
    out = []
    for fn, *args in specs:
        try:
            out += fn(*args)
        except Exception as e:
            out.append((f'{fn.__name__} crashed', False, f'{type(e).__name__}: {e}', True))
    return out

def report(checks):
    for name, ok, detail, hard in checks:
        print(f"{'PASS' if ok else ('FAIL' if hard else 'WARN')}  {name}" + (f'  [{detail}]' if detail else ''))
    return [c for c in checks if c[3] and not c[1]]

## 6-8. Pre-flight checks, data and resume

Cell 6 checks everything that can be checked before the GPUs start: the scripts and flags used here, the config, the packages (TensorFlow for the models, PyTorch for the eye masks), the data, the comparison model and the disk. Cell 7 builds the manifest, the cross-session split, the open-set list and the 20-class smoke tree, and writes the two image lists of the masks: every image to segment for training (`mask_list.csv`) and the ASA pool (`ASA_POOL_PER_CLASS` verification images per class, `asa_pool.csv`). Cell 8 does something only when `PREVIOUS_OUTPUT` is set.

In [ ]:
# 6. Pre-flight: GPUs, the merged code, its flags and the config, packages, data, comparison model, resume input, disk
import yaml
problems = []
n_gpu = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines())
print('GPUs visible:', n_gpu)
if n_gpu < 2:
    problems.append(f'{n_gpu} GPU(s) visible: set the accelerator to GPU T4 x2')
unknown = [a for a in ARCHS if a not in ('baseline', 'gap', 'attn', 'attn_mask', 'parts', 'attnet_gate')]
if unknown or len(set(ARCHS)) != len(ARCHS):
    problems.append(f'ARCHS: unknown or repeated archs {unknown or ARCHS}')
if ASA_ARCH != 'auto' and ASA_ARCH not in ARCHS:
    problems.append(f'ASA_ARCH = {ASA_ARCH!r} is not in ARCHS')
if RESUME_TEST_ARCH not in ARCHS:
    problems.append(f'RESUME_TEST_ARCH = {RESUME_TEST_ARCH!r} is not in ARCHS')
bad = [a for a in MASKING_ARCHS if a not in ('baseline', 'gap', 'attn', 'attn_mask', 'parts')]
if bad:
    problems.append(f'MASKING_ARCHS: {bad} (24 takes two-input models only: not attnet_gate)')
if set(MASKING_ARCHS) - set(ARCHS):
    print(f'NOTE: MASKING_ARCHS {sorted(set(MASKING_ARCHS) - set(ARCHS))} are not trained here: no 24 for them')

scripts = sorted({t for cmd in CMD.values() for t in cmd.split() if t.startswith('scripts/')})
missing = [p for p in scripts + [CFG] if not os.path.exists(f'{REPO}/{p}')]
if missing:
    problems.append(f'{BRANCH} lacks {missing}: merge the anatomy build first (or set BRANCH to a branch that has it)')
for path in scripts:
    if os.path.exists(f'{REPO}/{path}'):
        text = open(f'{REPO}/{path}').read()
        used = {t for cmd in CMD.values() if path in cmd.split()
                for t in cmd.split()[cmd.split().index(path) + 1:] if t.startswith('--')}
        lost = sorted(f for f in used if f'"{f}"' not in text and f"'{f}'" not in text)
        if lost:
            problems.append(f'{path} has no {lost}: merge the anatomy build first')
MAX_EPOCHS = None
if os.path.exists(f'{REPO}/{CFG}'):
    cfg = yaml.safe_load(open(f'{REPO}/{CFG}'))
    MAX_EPOCHS = int(cfg['training']['epochs'])
    print(f'{CFG}: training', cfg['training'], '| anatomy', cfg.get('anatomy'))
    if 'anatomy' not in cfg:
        problems.append(f'{CFG} has no anatomy section')
    cs = yaml.safe_load(open(f'{REPO}/configs/cross_session.yml'))
    differ = {k: (cs[k], cfg.get(k)) for k in cs if cs[k] != cfg.get(k)}
    if differ:
        print('NOTE: the recipe differs from configs/cross_session.yml (the baseline\'s):', differ)

code, tfv = py("import os; os.environ['TF_USE_LEGACY_KERAS'] = '1'; import importlib.metadata as m, tensorflow as tf; "
               "print(tf.__version__, m.version('tf-keras'), len(tf.config.list_physical_devices('GPU')))")
print('tensorflow, tf-keras, GPUs seen by TensorFlow:', tfv)
parts = tfv.split()
if code or len(parts) != 3 or not parts[0].startswith(('2.20.', '2.21.')) or parts[1] != '2.20.1':
    problems.append(f'TensorFlow 2.20 or 2.21 with tf-keras 2.20.1 expected: {tfv}')
code, torchv = py("import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())")
print('torch, torchvision, CUDA (17 and 21):', torchv)
if code:
    problems.append(f'PyTorch and torchvision are needed for the eye masks (17, 21): {torchv}')

VISOB = next((os.path.join(r, 'VISOB_Augmented_Recent') for r, dirs, _ in os.walk('/kaggle/input')
              if 'VISOB_Augmented_Recent' in dirs), None)
print('VISOB data:', VISOB)
if VISOB is None:
    problems.append('VISOB_Augmented_Recent not found under /kaggle/input: add mubashirulhassan00/visob-augmented-recent')
if BASELINE_CKPT:
    for path in (BASELINE_CKPT, class_index(BASELINE_CKPT)):
        if not os.path.exists(path):
            problems.append(f'BASELINE_CKPT: {path} not found (attach the output of train_cross_session.ipynb)')
elif 'baseline' not in ARCHS:
    print('NOTE: no BASELINE_CKPT and no baseline in ARCHS: 22 explains the mask model only')
PREV = None
if PREVIOUS_OUTPUT:
    found = sorted(glob.glob(f"{PREVIOUS_OUTPUT.rstrip('/')}/**/trained_models", recursive=True))
    PREV = os.path.dirname(found[0]) if len(found) == 1 else None
    print('earlier output:', PREV, '| trained_models folders found:', found)
    if PREV is None or not os.path.isdir(f'{PREV}/splits'):
        problems.append(f'PREVIOUS_OUTPUT = {PREVIOUS_OUTPUT}: expected exactly one trained_models folder with a splits '
                        f'folder next to it')
free = shutil.disk_usage(W).free / 2**30
print(f'free space in {W}: {free:.1f} GB')
need = 4 + 1.0 * len(ARCHS) * len(SEEDS) + (3 * len(SEEDS) if 'baseline' in ARCHS else 0)
if free < need:     # smoke models, masks; per head model about 0.4 GB + backup, the vendored baseline 1.7 GB + backup
    problems.append(f'only {free:.1f} GB free in {W} ({need:.0f} GB needed)')
print('\n'.join(['PROBLEMS:'] + problems) if problems else f'pre-flight ok ({hours():.2f} h into the session)')
assert not problems, 'fix the problems above, then run this cell again'

In [ ]:
# 7. Manifest, the cross-session split, the open-set list, the smoke tree and the mask image lists; checks
t0 = time.time()
FG['01_manifest'] = sh(CMD['01_manifest'])            # about 15 min
FG['02_cross_session'] = sh(CMD['02_cross_session'])  # tree /tmp/ocular_cross_session, manifest_split_cross_session.csv
FG['02_open_set'] = sh(CMD['02_open_set'])            # splits/open_set_test.csv and manifest_open_set.csv (P3)
FG['02_small'] = sh(CMD['02_small'])                  # the first SMOKE_CLASSES classes, for the smoke runs
print(f'data steps: {(time.time() - t0) / 60:.1f} min')
raw = pd.read_csv(f'{W}/manifest_raw.csv')
assert len(raw) == 134258, 'the VISOB copy is not the version the baseline was trained on (134,258 images): stop'
for name, manifest in (('cross-session', CS_MANIFEST), ('smoke tree', SMALL_MANIFEST)):
    print(f'\n{name}: images per split and phase\n{split_counts(manifest).to_string()}')
COUNTS = {name: pd.read_csv(manifest).split.value_counts().to_dict()
          for name, manifest in (('full', CS_MANIFEST), ('small', SMALL_MANIFEST))}
failed = report(run_checks((checks_split, CS_MANIFEST, N_CLASSES), (checks_split, SMALL_MANIFEST, SMOKE_CLASSES)))
assert not failed, 'the split is not what the protocol promises: stop'

# the images 21 segments for training: the tree (train, validation, test), plus the open set for attnet_gate's P3
cs = pd.read_csv(CS_MANIFEST)[['relpath', 'link_path', 'split']]
lists = [cs]
if 'attnet_gate' in ARCHS:
    lists.append(pd.read_csv(OPEN_MANIFEST)[['relpath', 'link_path']].assign(split='open_set'))
pd.concat(lists, ignore_index=True).to_csv(MASK_LIST, index=False)
MASK_COUNTS = pd.read_csv(MASK_LIST).split.value_counts().to_dict()
# the ASA pool: a fixed sample of verification (test) images per class, segmented at 256 x 256 by 17 for 22
test = pd.read_csv(f'{SPLITS}/cross_session_test.csv')
pool = test.sample(frac=1.0, random_state=0).groupby('subject').head(ASA_POOL_PER_CLASS).sort_values('relpath')
pool.to_csv(ASA_POOL, index=False)
print(f'mask list: {MASK_COUNTS} | ASA pool: {len(pool)} verification images of {pool.subject.nunique()} classes')

In [ ]:
# 8. Resume (only with PREVIOUS_OUTPUT): copy the earlier models, masks and results; the split must be identical
def training_finished(t, models=None):
    # 03 writes run_<tag>.json with finished = true when training ends and then removes the backup
    models = models or MODELS
    path = f'{models}/run_{t}.json'
    return os.path.exists(path) and load(path).get('finished') is True and not os.path.isdir(f'{models}/backup_{t}')

def training_done(t, models=None):
    # finished AND 03's last stage written: the weights (with their sha256 in the record) and results_<tag>.json.
    # This is 03's own test: a finished run without them is not done, and 03 --resume writes them (a stop
    # during 03's test evaluation leaves exactly this state), so such a step must be launched again
    models = models or MODELS
    if not training_finished(t, models):
        return False
    return (os.path.exists(f'{models}/results_{t}.json') and os.path.exists(f'{models}/Xception_ArcFace_{t}.weights.h5')
            and bool(load(f'{models}/run_{t}.json').get('weights_sha256')))

def epochs_logged(t):
    return csv_rows(f'{MODELS}/log_{t}.csv')

def masks_done(t, wanted=None):
    # the masks64 file and its summary exist and, with wanted ({split: images}), cover every wanted split with
    # exactly that many images: masks copied from a session that ran without attnet_gate have no open set
    summary = f'{TRAIN_MASKS}/masks64_summary_{t}.json'
    if not (os.path.exists(f'{TRAIN_MASKS}/masks64_{t}.npz') and os.path.exists(summary)):
        return False
    per = load(summary).get('images_per_split') or {}
    return wanted is None or all(per.get(k, 0) == n for k, n in wanted.items())

if PREV:
    for split in ('train', 'validation', 'test'):
        old, new = pd.read_csv(f'{PREV}/splits/cross_session_{split}.csv'), pd.read_csv(f'{SPLITS}/cross_session_{split}.csv')
        assert old.equals(new), f'the {split} split differs from the earlier session: resuming would train on other images'
    for folder in ('trained_models', 'eval_clean', 'verification', 'train_masks', 'eye_masks', 'asa', 'analysis_asa',
                   'masking'):
        if os.path.isdir(f'{PREV}/{folder}'):
            shutil.copytree(f'{PREV}/{folder}', f'{W}/{folder}', copy_function=shutil.copyfile, dirs_exist_ok=True)
    subprocess.run(['chmod', '-R', 'u+w', W], capture_output=True)
    if os.path.isdir(f'{PREV}/logs'):
        shutil.copytree(f'{PREV}/logs', f'{LOGS}/earlier_session', copy_function=shutil.copyfile, dirs_exist_ok=True)
    print(f'21 masks: {masks_done(TAG)} | they cover this session\'s mask list {MASK_COUNTS}: '
          f'{masks_done(TAG, MASK_COUNTS)} | 17 ASA pool masks: {os.path.exists(POOL_MASKS)}')
    if masks_done(TAG) and not masks_done(TAG, MASK_COUNTS):
        print(f"  the earlier masks cover {load(f'{TRAIN_MASKS}/masks64_summary_{TAG}.json').get('images_per_split')}: "
              f'21 runs again on the whole mask list (e.g. attnet_gate was added and needs the open set)')
    for (a, s), t in FULL_TAGS.items():
        state = ('finished' if training_done(t) else
                 'finished, weights or test results missing (03 --resume writes them)' if training_finished(t) else
                 f'backup after {epochs_logged(t)} epochs' if os.path.isdir(f'{MODELS}/backup_{t}') else 'not started')
        print(f'{t}: training {state} | 04 done: {os.path.exists(f"{EVAL}/summary_{t}.json")} | '
              f'15 done: {os.path.exists(f"{VERIF}/verification_summary_{t}.json")}'
              + (f' | 24 done: {os.path.exists(f"{MASKING}/summary_masking_{t}.json")}' if f'24_{t}' in CMD else ''))
else:
    print('PREVIOUS_OUTPUT is None: everything starts from the beginning')

## 9-11. Smoke runs, checks and the time estimate

Cell 9 segments the smoke tree (21) and the ASA pool (17), and trains every arch for 2 epochs on the 20-class tree, both GPUs busy as in the full run (each mask arch waits for the smoke masks). Cell 10 tests `--resume` with `RESUME_TEST_ARCH` (stopped as soon as the backup of its first epoch is complete, then run again) while the other GPU runs 04 and 15 on every smoke model and 24 (`MASKING_SMOKE_IMAGES` images, the full run's grids) on the smoke models of `MASKING_ARCHS`; then 22 and 23 run on a mask smoke model with the ASA pool (smoke C, with the full run's 22 flags). Cell 11 checks every output and estimates the full run: per arch the steady smoke epoch scaled by the training images (the checkpoint and backup writes are not scaled), 21 from its smoke rate, 04 and 15 from their smoke runs, 24 per batch of 16 images from its smoke run on that arch (one full batch), and 22 per image from its smoke run. Masking flags of the smoke models (2 epochs on 20 classes) are only warnings. It refuses the full runs if they would not finish before `HARD_STOP_H` (see `START_IF_TOO_LONG`).

In [ ]:
# 9. Smoke A: masks of the smoke tree (21) and the ASA pool (17), then every arch's smoke training, on both GPUs
if os.path.isdir(SMOKE_MODELS):
    shutil.rmtree(SMOKE_MODELS)       # a leftover backup would make the smoke trainings resume instead of start
first = ['21_smoke'] + [f'03_{st}' for a, st in SMOKE_TAGS.items() if a in MASK_ARCHS]
second = ([] if os.path.exists(POOL_MASKS) else ['17_pool']) + \
    [f'03_{st}' for a, st in SMOKE_TAGS.items() if a not in MASK_ARCHS]
SA = wait(launch_all({0: first, 1: second}))
for a, st in SMOKE_TAGS.items():
    print(f'{st}: epoch seconds {epoch_seconds(f"03_{st}")} | step {seconds(f"03_{st}")} s')

In [ ]:
# 10. Smoke B: the --resume test on GPU 0; 04 and 15 on every smoke model on GPU 1. Smoke C: 22 and 23
for path in glob.glob(f'{SMOKE_MODELS}/*{SMOKE}_resume*'):   # a rerun of this cell starts the test afresh
    shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
gpu1 = launch_all({1: [f'{step}_{st}' for st in SMOKE_TAGS.values() for step in ('04', '15')]
                       + [f'24_{st}' for st in SMOKE_TAGS.values() if f'24_{st}' in CMD]})
first = launch('GPU 0', ['03_resume_a'], gpu=0)
rt, backup, rows_at_stop, WRITE_S = f'{SMOKE}_resume', f'{SMOKE_MODELS}/backup_{SMOKE}_resume', None, None
while first['proc'].poll() is None and hours() < HARD_STOP_H:
    time.sleep(5)
    if csv_rows(f'{SMOKE_MODELS}/log_{rt}.csv') >= 1 and backup_written(backup):
        rows_at_stop = csv_rows(f'{SMOKE_MODELS}/log_{rt}.csv')
        backup_s = max(mtime(p) for p in glob.glob(f'{backup}/**/checkpoint', recursive=True)) \
            - mtime(f'{SMOKE_MODELS}/log_{rt}.csv')
        WRITE_S = 2 * backup_s if backup_s >= 0 else None     # checkpoint + backup writes per epoch
        print(f'stopping the first run after {rows_at_stop} logged epoch(s), as the 12-hour limit would '
              f'(checkpoint + backup writes: about {WRITE_S} s per epoch)')
        stop([first])
        break
if rows_at_stop is None:
    print('the first run ended before its first backup was complete:', status('03_resume_a'))
    print('\n'.join(tail(f'{LOGS}/03_resume_a.log', 40)))
SB = wait([launch('GPU 0', ['03_resume_b'], gpu=0)], also=gpu1) if rows_at_stop is not None else {}
SB.update(wait(gpu1))

# smoke C: 22 on the smoke model of SMOKE_ASA_ARCH with pool images it classifies correctly (all pool images of the
# smoke classes if fewer than ASA_SMOKE_IMAGES, or, with ASA_VERIF_PAIRS, fewer than two classes for the impostor
# templates), then 23 on its rows
st = SMOKE_TAGS[SMOKE_ASA_ARCH]
pool_rel = set(pd.read_csv(ASA_POOL).relpath)
if os.path.exists(f'{EVAL}/predictions_{st}.csv'):
    pred = pd.read_csv(f'{EVAL}/predictions_{st}.csv')
    pred = pred[pred.relpath.isin(pool_rel)]
    chosen = with_templates(pred[pred.correct])
    if len(chosen) < ASA_SMOKE_IMAGES or (ASA_VERIF_PAIRS and chosen.true_idx.nunique() < 2):
        chosen = with_templates(pred)
    chosen.to_csv(ASA_SMOKE_LIST, index=False)
    print(f'22 smoke: {len(chosen)} pool images ({int(chosen.correct.sum())} classified correctly by {st})')
    SC = {'22_smoke': sh(CMD['22_smoke'], check=False, gpu=0)}
    SC['23_smoke'] = sh(CMD['23_smoke'], check=False) if SC['22_smoke'] == 0 else 'skipped'
else:
    SC = {'22_smoke': 'skipped (no 04 smoke predictions)', '23_smoke': 'skipped'}
print(SC)

In [ ]:
# 11. Smoke checks, the time estimate for the full runs and the 12-hour check
smoke_failed = report(run_checks(
    (checks_masks, SMOKE, COUNTS['small']),
    *[(checks_03, st, a, SMOKE_MODELS, SMOKE_CLASSES, SMOKE_EPOCHS, SEEDS[0], SMALL_MASKS64) for a, st in SMOKE_TAGS.items()],
    (checks_resume, rows_at_stop),
    *[(checks_04, st, SMALL_MANIFEST, a) for a, st in SMOKE_TAGS.items()],
    *[(checks_15, st, ('p2',) if a in TEST_MASK_ARCHS else ('p2', 'p3')) for a, st in SMOKE_TAGS.items()],
    *[(checks_masking, st) for st in SMOKE_TAGS.values() if f'24_{st}' in CMD],
    (checks_asa, SMOKE)))
assert all(SA.get(f'03_{st}') == 0 for st in SMOKE_TAGS.values()), '03 failed in the smoke run: fix it first'
assert SB.get('03_resume_b') == 0, '--resume did not work in the smoke test: a stopped run could not be continued'
assert all(SB.get(f'{s}_{st}') == 0 for st in SMOKE_TAGS.values() for s in ('04', '15')), '04 or 15 failed: see above'
assert all(SB.get(f'24_{st}') == 0 for st in SMOKE_TAGS.values() if f'24_{st}' in CMD), '24 failed: see above'
assert SC.get('22_smoke') == 0 and SC.get('23_smoke') == 0, '22 or 23 failed in the smoke run: see above'
assert not smoke_failed, 'fix the failed checks above before the full runs'

full, small = COUNTS['full'], COUNTS['small']
ratio = max(full['train'] / small['train'], full.get('validation', 0) / max(small.get('validation', 1), 1))
fixed_s = WRITE_S or 0
EST = {'epoch_min': {}, 'model_h': {}}
for a, st in SMOKE_TAGS.items():
    e = epoch_seconds(f'03_{st}')
    assert e, f'no epoch times in the 03 smoke log of {a} (03 must run with --verbose 2)'
    fixed = min(fixed_s, e[-1])
    EST['epoch_min'][a] = ((e[-1] - fixed) * ratio + fixed) / 60
# 04 and 15 from the smoke runs (one image rate for every arch: the backbone dominates), 21 from its smoke rate
per_img_04 = max(seconds(f'04_{SMOKE_TAGS[ARCHS[0]]}') or 60, 1) / small['test']
EST['04_h'] = per_img_04 * full['test'] / 3600
v = load(f'{VERIF}/verification_summary_{SMOKE_TAGS[ARCHS[0]]}.json')
kept = raw if MAX_FRAME is None else raw[raw.idx <= MAX_FRAME]   # 15 --max-frame leaves the rest out
n15 = int((kept.subject.between(1141, 3279) | (kept.subject == 7000)).sum()) + len(pd.read_csv(OPEN_SET))
EST['15_h'] = (v['timing']['load_model_s'] + v['timing']['embedding_s'] / v['n_images_embedded'] * n15 + 600) / 3600
m21 = load(f'{TRAIN_MASKS}/masks64_summary_{SMOKE}.json')
rate21 = m21['timing'].get('segmenting_s', m21['timing']['total_s']) / m21['n_images']
EST['21_h'] = 0 if masks_done(TAG, MASK_COUNTS) else (rate21 * sum(MASK_COUNTS.values()) + 300) / 3600
def masking_hours(s):
    # 24's full run (one image per class, at most MASKING_MAX_IMAGES, plus loading the model) from its smoke run:
    # per batch when the smoke run filled a whole batch (a batch of 4 images costs about as much as one of 16, so
    # a per-image rate overstates the full run), else per image (too high, never too low)
    n = min(MASKING_MAX_IMAGES or N_CLASSES, N_CLASSES)
    if s.get('seconds_per_batch') and s['n_images'] >= s['batch_size']:
        seconds = s['seconds_per_batch'] * -(-n // s['batch_size'])
    else:
        seconds = s['seconds_per_image'] * n
    return (seconds + 300) / 3600
EST['24_h'] = {a: masking_hours(load(f'{MASKING}/summary_masking_{SMOKE_TAGS[a]}.json')) for a in MASK_CHECK}
for (a, s), t in FULL_TAGS.items():
    left = 0 if training_finished(t) else MAX_EPOCHS - epochs_logged(t)
    h = left * EST['epoch_min'][a] / 60 + (0 if training_done(t) else 0.3)   # + start, model build, 03's test evaluation
    h += 0 if os.path.exists(f'{EVAL}/summary_{t}.json') else EST['04_h']
    h += 0 if os.path.exists(f'{VERIF}/verification_summary_{t}.json') else EST['15_h']
    if f'24_{t}' in CMD:
        h += 0 if os.path.exists(f'{MASKING}/summary_masking_{t}.json') else EST['24_h'][a]
    EST['model_h'][t] = h
a22 = load(f'{ASA}/summary_asa_{SMOKE}.json')
EST['22_min_per_image'] = a22['minutes'] / max(a22['n_images'], 1)     # one run alone, ASA_WORKERS threads
EST['22_pair_factor'] = ASA_PAIR_FACTOR if BASE else 1.0                # cell 13 runs two models at once
n_asa = ASA_MAX_IMAGES or N_CLASSES
EST['22_h'] = (2 * ASA_SMOKE_IMAGES + n_asa) * EST['22_min_per_image'] * EST['22_pair_factor'] / 60
jobs = [(a, [f'{step}_{t}' for step in MODEL_STEPS if f'{step}_{t}' in CMD], EST['model_h'][t])
        for (a, s), t in FULL_TAGS.items()]
needs21 = any(a in MASK_ARCHS for a, _, _ in jobs) and not masks_done(TAG, MASK_COUNTS)
CHAINS, LOAD = plan_chains(jobs, ['21_full'] if needs21 else [], EST['21_h'] if needs21 else 0.0)
EST['per_gpu_h'] = LOAD
finish = hours() + max(LOAD.values()) + 0.2
print(f"full epoch per arch (min, cap {MAX_EPOCHS} epochs): {', '.join(f'{a} {m:.1f}' for a, m in EST['epoch_min'].items())}")
print(f"21 (all {sum(MASK_COUNTS.values())} images): {EST['21_h'] * 60:.0f} min | 04: {EST['04_h'] * 60:.0f} min | "
      f"15: {EST['15_h'] * 60:.0f} min per model | 24: "
      + (', '.join(f'{a} {h * 60:.0f} min' for a, h in EST['24_h'].items()) or 'none'))
print('per model, what is left (h, all epochs to the cap):', {t: round(h, 1) for t, h in EST['model_h'].items()})
for g, names in CHAINS.items():
    print(f'GPU {g}: {LOAD[g]:.1f} h: {" -> ".join(names)}')
print(f"trainings end at about {finish:.1f} h of the 12 (HARD_STOP_H = {HARD_STOP_H}); 22 afterwards: about "
      f"{EST['22_h']:.1f} h for {n_asa} images per model on two GPUs ({EST['22_min_per_image']:.2f} min per image "
      f"alone, x {EST['22_pair_factor']} for two runs sharing the CPUs; 22 resumes in the next session if it is stopped)")
if RUN_FULL:
    for path in glob.glob(f'{SMOKE_MODELS}/*.keras') + glob.glob(f'{SMOKE_MODELS}/*.h5'):
        os.remove(path)               # room for the full runs; the smoke records stay
if RUN_FULL and finish >= HARD_STOP_H:
    assert START_IF_TOO_LONG, ('the trainings might not finish in this session: set START_IF_TOO_LONG = True to train '
                               'until HARD_STOP_H and resume in a new session (see Resume), or train fewer archs or seeds')
    print('WARNING: the estimate ends after HARD_STOP_H; START_IF_TOO_LONG is True, so the runs start and continue '
          'in the next session')

## 12-13. The full runs (`RUN_FULL = True`)

Cell 12 runs the chains planned in cell 11: GPU 0 first segments every image of the tree (21, masks64; plus the open set if `attnet_gate` is trained) while GPU 1 trains the mask-free archs; each mask model waits for the masks. Every model trains with `configs/anatomy.yml`, `--resume`, `--reshuffle` and `--save-weights`, then 04 evaluates it on the cross-session test split (every verification image) and 15 runs P2 and P3; `attnet_gate` gets its region input from the same masks64 file (`--region-masks`). The models of `MASKING_ARCHS` (seed `SEEDS[0]`) then get 24 on one correctly classified image per class. Steps an earlier session finished are skipped (a 03 step only when its weights and test results exist: otherwise 03 `--resume` writes them). Cell 13 picks the mask model for 22 (`ASA_ARCH`), runs 04 on `BASELINE_CKPT` if its correct-image list is missing and there is time, takes the pool images that the mask model and the baseline both classify correctly (the same images for both models), and runs 22 (smoke, then the full run without the smoke images, `--resume`) on one GPU each, then 23. A 22 run stopped at `HARD_STOP_H` keeps its finished batches and continues in the next session.

In [ ]:
# 12. Full runs (RUN_FULL): 21, then per model 03 -> 04 -> 15 P2 + P3 (-> 24), queued over both GPUs; finished
# steps skipped
def mark_done(name, why):
    with open(f'{LOGS}/{name}.log', 'w') as f:
        f.write(why + '\n')
    with open(f'{LOGS}/{name}.exit', 'w') as f:
        f.write('0\n')

C = {}
if RUN_FULL:
    if not needs21:
        mark_done('21_full', f'not needed in this session (no mask model left to train, or {MASKS64} covers the '
                             f'mask list)')
    chains = {}
    for g, names in CHAINS.items():
        keep = []
        for name in names:
            step, t = name.split('_', 1)
            out = {'03': f'{MODELS}/run_{t}.json', '04': f'{EVAL}/summary_{t}.json',
                   '15': f'{VERIF}/verification_summary_{t}.json', '24': f'{MASKING}/summary_masking_{t}.json'}.get(step)
            # training_done: finished with weights and test results, so a 03 step stopped during its test
            # evaluation is launched again (03 --resume then writes only the missing files)
            if out and step != '21' and training_done(t) and os.path.exists(out):
                mark_done(name, f'done in an earlier session ({out})')
            else:
                keep.append(name)
        chains[g] = keep
    C = wait(launch_all(chains), every=300) if any(chains.values()) else {}
else:
    print('RUN_FULL is False: full runs skipped. Set RUN_FULL = True (cell 2) and use Save Version > Save & Run All.')

In [ ]:
# 13. ASA (RUN_FULL): 22 on the mask model and the baseline (same images), smoke then full, one GPU each; then 23
A = {}
if RUN_FULL:
    if ASA_ARCH == 'auto':            # the mask-supervised model with the lowest P2 EER
        eer = {}
        for a in LOSS_ARCHS:
            path = f'{VERIF}/verification_summary_{tag(a, SEEDS[0])}.json'
            if a in ARCHS and os.path.exists(path):
                eer[a] = load(path)['per_mode']['p2']['eer']
        ASA_MASK_ARCH = min(eer, key=eer.get) if eer else ASA_MASK_ARCH
        print('P2 EER of the mask models:', eer, '-> 22 explains', ASA_MASK_ARCH)
    CMD.update(asa_cmds('mask', ckpt(MODELS, tag(ASA_MASK_ARCH, SEEDS[0]))))
    models = {'mask': (ckpt(MODELS, tag(ASA_MASK_ARCH, SEEDS[0])), f'{EVAL}/correct_test_{tag(ASA_MASK_ARCH, SEEDS[0])}.csv')}
    mask_ready = all(os.path.exists(p) for p in models['mask'])
    if BASE:
        base_tag = 'base_ext' if BASELINE_CKPT else tag('baseline', SEEDS[0])
        models['base'] = (BASE, f'{EVAL}/correct_test_{base_tag}.csv')
        if BASELINE_CKPT and not os.path.exists(f'{EVAL}/summary_base_ext.json'):
            # 04 on the external baseline (its correct-image list) runs in the foreground: only when 22 could use
            # it (the mask model's list exists) and it ends before HARD_STOP_H with room for 22 to start
            need04 = EST['04_h'] + 0.25
            if not mask_ready:
                print('04 on BASELINE_CKPT skipped: the mask model has no checkpoint or correct-image list yet, '
                      'so 22 cannot run in this session')
            elif hours() + need04 >= HARD_STOP_H:
                print(f'04 on BASELINE_CKPT skipped: about {need04:.1f} h needed, {HARD_STOP_H - hours():.1f} h left '
                      f'before HARD_STOP_H; it runs in the next session (PREVIOUS_OUTPUT)')
            else:
                A['04_base_ext'] = sh(CMD['04_base_ext'], check=False, gpu=0)
    missing = [p for path, csv in models.values() for p in (path, csv) if not os.path.exists(p)]
    if missing:
        print('22 skipped: missing', missing)
    else:
        pool_rel = set(pd.read_csv(ASA_POOL).relpath)
        lists = [pd.read_csv(csv) for _, csv in models.values()]
        both = set.intersection(*[set(l.relpath) for l in lists]) & pool_rel
        chosen = with_templates(lists[0][lists[0].relpath.isin(both)])
        chosen.to_csv(ASA_LIST, index=False)
        print(f'22 images: {len(chosen)} pool images of {chosen.true_idx.nunique()} classes, correct for '
              f'{" and ".join(models)}' + (' (classes with a second such image: the genuine templates)'
                                          if ASA_VERIF_PAIRS else ''))
        n_asa = min(ASA_MAX_IMAGES or N_CLASSES, chosen.true_idx.nunique())
        names = list(models)
        # minutes per image of each run: the smoke rate (one run alone) times the allowance for two runs in parallel
        rate = EST['22_min_per_image'] * (ASA_PAIR_FACTOR if len(names) > 1 else 1.0)
        done = {n: (load(f'{ASA}/parts_asa_{TAG}_{n}/state.json')['n_done']
                    if os.path.exists(f'{ASA}/parts_asa_{TAG}_{n}/state.json') else 0) for n in names}
        need_h = (ASA_SMOKE_IMAGES + n_asa - min(done.values())) * rate / 60 + 0.2
        first_h = (ASA_SMOKE_IMAGES + 16) * rate / 60 + 0.1                 # the smoke run and one batch of 22
        if chosen.true_idx.nunique() <= ASA_SMOKE_IMAGES:
            print(f'22 skipped: only {chosen.true_idx.nunique()} classes have a pool image that every model classifies '
                  f'correctly (raise ASA_POOL_PER_CLASS)')
        elif hours() + first_h >= HARD_STOP_H:
            print(f'22 skipped: {HARD_STOP_H - hours():.1f} h left before HARD_STOP_H, not enough for its smoke run and '
                  f'one batch. Run the next session with PREVIOUS_OUTPUT: the smoke runs of cells 9-11 run again '
                  f'(about 1 h), finished trainings and evaluations are skipped, then 22 starts (or continues).')
        else:
            if hours() + need_h >= HARD_STOP_H:
                print(f'22: about {need_h:.1f} h left to run, {HARD_STOP_H - hours():.1f} h before HARD_STOP_H: it is '
                      f'stopped there with its finished batches kept, and continues in the next session (--resume)')
            for name in names:
                CMD.update(asa_cmds(name, models[name][0], ASA_LIST))
            A.update(wait(launch_all({g: [f'22_smoke_{n}', f'22_{n}'] for g, n in enumerate(names)}), every=120))
            for name in names:
                if A.get(f'22_{name}') == 0:
                    A[f'23_{name}'] = sh(CMD[f'23_{name}'], check=False)
else:
    print('RUN_FULL is False: 22 skipped.')

## 14-16. Results table, results.zip and the final checklist

Cell 14 collects one row per model (arch, seed, epochs, best epoch, test accuracy, P2 and P3 EER, TAR at FAR 1e-2 and 1e-3, rank-1, mask coverage, the masking flags of 24). `results.zip` (cell 15) is written before the checklist, so a failed check never loses the outputs; it holds records, logs, split lists, tables and plots, but **no checkpoints, no masks and no images** (17's QC overlays stay out). The checklist (cell 16) asserts the expected files and the checks above.

In [ ]:
# 14. One row per model -> summary_anatomy.csv
rows = []
for (a, s), t in FULL_TAGS.items():
    row = {'arch': a, 'seed': s, 'tag': t}
    if os.path.exists(f'{MODELS}/run_{t}.json'):
        run = load(f'{MODELS}/run_{t}.json')
        row.update({k: run.get(k) for k in ('epochs_run', 'best_epoch', 'best_val_loss', 'stopped_early',
                                            'minutes_all_sessions', 'resumed')})
        cov = run.get('mask_coverage') or {}
        row['mask_coverage_train'] = (cov.get('train') or {}).get('share')
    if os.path.exists(f'{EVAL}/summary_{t}.json'):
        row['test_acc_margin_free'] = load(f'{EVAL}/summary_{t}.json')['accuracy_margin_free']
    if os.path.exists(f'{VERIF}/verification_summary_{t}.json'):
        for mode, r in load(f'{VERIF}/verification_summary_{t}.json')['per_mode'].items():
            for k in ('eer', 'tar_at_far_1e-2', 'tar_at_far_1e-3', 'rank1'):
                row[f'{mode}_{k}'] = r.get(k)
    if os.path.exists(f'{MASKING}/summary_masking_{t}.json'):
        row['masking_flags'] = ';'.join(load(f'{MASKING}/summary_masking_{t}.json')['masking_flags']) or 'none'
    rows.append(row)
table = pd.DataFrame(rows)
table.to_csv(f'{W}/summary_anatomy.csv', index=False)
with pd.option_context('display.width', 250, 'display.max_columns', 40):
    print(table.round(4).to_string(index=False))

In [ ]:
# 15. results.zip without checkpoints, masks or images (download it from the Output panel; never commit or upload it)
import zipfile
info = {'branch': BRANCH, 'commit': COMMIT, 'run_full': RUN_FULL, 'tag': TAG, 'smoke_tag': SMOKE,
        'settings': {'archs': ARCHS, 'seeds': SEEDS, 'max_frame': MAX_FRAME, 'cfg': CFG, 'max_epochs': MAX_EPOCHS,
                     'mask_loss_weight': MASK_LOSS_WEIGHT, 'baseline_ckpt': BASELINE_CKPT, 'asa_arch': ASA_ARCH,
                     'asa_mask_arch': ASA_MASK_ARCH, 'asa_pool_per_class': ASA_POOL_PER_CLASS,
                     'asa_max_images': ASA_MAX_IMAGES, 'asa_attacks': ASA_ATTACKS, 'asa_verif_pairs': ASA_VERIF_PAIRS,
                     'asa_baseline_hiding': ASA_BASELINE_HIDING, 'asa_pixel_conditions': ASA_PIXEL_CONDITIONS,
                     'masking_archs': MASKING_ARCHS, 'masking_max_images': MASKING_MAX_IMAGES, 'masking_smoke_images': MASKING_SMOKE_IMAGES,
                     'smoke_classes': SMOKE_CLASSES, 'smoke_epochs': SMOKE_EPOCHS,
                     'n_boot': N_BOOT, 'previous_output': PREVIOUS_OUTPUT, 'start_if_too_long': START_IF_TOO_LONG},
        'commands': CMD, 'steps': {**{n: status(n) for n in CMD if n not in FG}, **FG},
        'counts': COUNTS, 'mask_counts': MASK_COUNTS, 'estimates_h': globals().get('EST', {}),
        'chains': globals().get('CHAINS', {}), 'session_hours': round(hours(), 2)}
with open(f'{W}/run_info.json', 'w') as f:
    json.dump(info, f, indent=2, default=str)
left_out = []
with zipfile.ZipFile(f'{W}/results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('run_info.json', 'summary_anatomy.csv', 'asa_pool.csv', 'asa_images.csv'):
        if os.path.exists(f'{W}/{name}'):
            z.write(f'{W}/{name}', name)
    for folder in ('trained_models', 'smoke_models', 'eval_clean', 'verification', 'splits', 'logs', 'train_masks',
                   'eye_masks', 'asa', 'analysis_asa', 'masking'):
        for root, dirs, files in os.walk(f'{W}/{folder}'):
            dirs[:] = [d for d in dirs if not d.startswith('backup_')]
            for name in sorted(files):
                path = os.path.join(root, name)
                rel = os.path.relpath(path, W)
                if zip_keep(rel) and os.path.getsize(path) <= ZIP_MAX_MB * 2**20:
                    z.write(path, rel)
                else:
                    left_out.append(rel)
print(f"results.zip: {os.path.getsize(f'{W}/results.zip') / 2**20:.1f} MB")
print('\nleft out of results.zip (checkpoints, masks, QC images; they stay in the Output panel):')
for rel in left_out:
    print(f'  {rel}  ({os.path.getsize(f"{W}/{rel}") / 2**20:.1f} MB)')
print('\nKeep the checkpoints and masks in Kaggle only: attach this version\'s output to later notebooks or make a '
      'private dataset from it. Never make the notebook, its output or that dataset public.')

In [ ]:
# 16. Final checklist: expected files and sanity checks; the run fails here if a hard check fails
if RUN_FULL:
    full_masks = {k: v for k, v in MASK_COUNTS.items()}
    checks = run_checks((checks_split, CS_MANIFEST, N_CLASSES),
                        *([(checks_masks, TAG, full_masks)] if any(a in MASK_ARCHS for a in ARCHS) else []),
                        *[(checks_03, t, a, MODELS, N_CLASSES, MAX_EPOCHS, s, MASKS64) for (a, s), t in FULL_TAGS.items()],
                        *[(checks_04, t, CS_MANIFEST, a) for (a, s), t in FULL_TAGS.items()],
                        *[(checks_15, t) for t in FULL_TAGS.values()],
                        *[(checks_masking, t) for t in FULL_TAGS.values() if f'24_{t}' in CMD],
                        *[(checks_asa, f'{TAG}_{n}') for n in ('mask', 'base') if f'22_{n}' in A])
    expected = ['01_manifest', '02_cross_session', '02_open_set', '02_small', '21_smoke'] + \
        [f'{step}_{t}' for t in FULL_TAGS.values() for step in MODEL_STEPS if f'{step}_{t}' in CMD] + \
        [k for k in A if k.startswith(('22_', '23_'))]
else:
    checks = run_checks((checks_split, CS_MANIFEST, N_CLASSES), (checks_masks, SMOKE, COUNTS['small']),
                        *[(checks_03, st, a, SMOKE_MODELS, SMOKE_CLASSES, SMOKE_EPOCHS, SEEDS[0], SMALL_MASKS64)
                          for a, st in SMOKE_TAGS.items()],
                        *[(checks_masking, st) for st in SMOKE_TAGS.values() if f'24_{st}' in CMD],
                        (checks_resume, rows_at_stop), (checks_asa, SMOKE))
    expected = ['01_manifest', '02_cross_session', '02_open_set', '02_small', '21_smoke', '03_resume_b'] + \
        [f'{step}_{st}' for st in SMOKE_TAGS.values() for step in MODEL_STEPS if f'{step}_{st}' in CMD]
steps = {n: FG.get(n, A.get(n, SC.get(n, status(n)))) for n in expected}
bad = {n: s for n, s in steps.items() if s != 0}
checks.append(('every step of the session succeeded', not bad, ', '.join(f'{n}: {s}' for n, s in bad.items()), True))
checks.append(('results.zip written', os.path.exists(f'{W}/results.zip'), '', True))
failed = report(checks)
warned = [c for c in checks if not c[1] and not c[3]]
record = {'run_full': RUN_FULL, 'commit': COMMIT, 'session_hours': round(hours(), 2), 'steps': steps,
          'checks': [{'check': n, 'passed': bool(ok), 'detail': d, 'hard': h} for n, ok, d, h in checks]}
name = f"checklist_{TAG if RUN_FULL else SMOKE}.json"
with open(f'{W}/{name}', 'w') as f:
    json.dump(record, f, indent=2, default=str)
with zipfile.ZipFile(f'{W}/results.zip', 'a') as z:
    z.write(f'{W}/{name}', name)
print(f'\n{len(checks) - len(failed) - len(warned)} passed, {len(warned)} warnings, {len(failed)} failed | '
      f'session {hours():.1f} h')
if RUN_FULL and glob.glob(f'{MODELS}/backup_*'):
    print('A training was stopped before it finished: resume it in a new session (set PREVIOUS_OUTPUT, see the top).')
assert not failed, 'some checks failed (listed above); the outputs and results.zip are saved'

## After the run

1. Download `results.zip` and unzip it into a new folder under `D:\TempCLG\results` on the laptop. Never commit it or put it on the web; back it up in the team's private storage.
2. Keep the checkpoints (`trained_models/Xception_ArcFace_an_<arch>_s<seed>.keras` and the `.weights.h5` copies) and the masks (`train_masks/masks64_an.npz`, `eye_masks/masks_asa_pool.npz`) in Kaggle: attach this version's output to later notebooks, or make a **private** dataset from it.
3. Compare the models on the cross-session numbers (`summary_anatomy.csv`, the CIs in `verification/`) against the cross-session baseline of `train_cross_session.ipynb`, never against the 97.96 % of the random split. `attnet_gate` used the eye masks at test time; the other models did not.
4. The part presence of `train_masks/part_presence_an.csv` says how often each part is in the crop at all (e.g. the eyebrow): read the parts model's receipt and ASA values for a part only on images where it is present.
5. The masking check (`masking/summary_masking_an_<arch>_s1.json`, column `masking_flags` of `summary_anatomy.csv`): a robustness difference between two models counts only if neither has a flag. Flags such as `pgd_budget_not_monotone`, `unbounded_pgd_below_0.99` or `aas_pgd_beats_pgd@<L>levels` mean the attacks of that model are not trustworthy as they stand: use `aas_pgd` (22, 14 `--attacks ...,aas_pgd`) or more steps and restarts before comparing.
6. The ASA results (`analysis_asa/an_mask/report_asa.md`, `analysis_asa/an_base/report_asa.md`) explain the same images for both models, with the KernelSHAP / LIME baselines, the verification pairs, the run times and the agreement of the maps (23). If 22 was skipped or stopped for time, run the next session with `PREVIOUS_OUTPUT`: the smoke runs of cells 9-11 run again (about 1 h), finished trainings and evaluations are skipped, and 22 starts or continues after its last finished batch.